**Autores:**

- Gonçalo Henriques (123422)
- Rodrigo Sousa (123390)

In [1]:
# Importações básicas
from pyspark.sql import SparkSession
from pymongo import MongoClient
from pyspark.sql import functions as F
import os
import subprocess
import shutil
from pathlib import Path
from getpass import getpass
from functools import reduce
from pyspark.sql.types import StringType

In [2]:
# Criar SparkSession
spark = SparkSession.builder \
    .appName("LondonBikesDataIngestion") \
    .getOrCreate()

spark

**- Ligação ao serviço MongoDB a correr no ambiente Docker Compose...**

In [3]:
# Dentro do Docker, o container MongoDB é acessível pelo nome do serviço: "mongodb"
client = MongoClient("mongodb://mongodb:27017/")

# Selecionar a base de dados utilizada neste projeto
db = client["london_bikes"]

# Verificar se a ligação ao MongoDB está ativa
client.admin.command("ping")

client

MongoClient(host=['mongodb:27017'], document_class=dict, tz_aware=False, connect=True)

---
# **1 - Ingestão de Dados**

Este notebook prepara o dataset de bicicletas de Londres para as etapas seguintes do projeto.

Os ficheiros CSV podem ser obtidos de duas formas:

1. **Kaggle API**: utilizar um token da API do Kaggle e descarregar o dataset diretamente a partir do notebook.
2. **Download manual**: descarregar o dataset localmente a partir da página do Kaggle e colocar/extrair os ficheiros CSV em `Projeto_Final/archive`.

Fonte do dataset:

https://www.kaggle.com/datasets/ioexception/london-cycles/data

Se os ficheiros já existirem em `Projeto_Final/archive`, a célula de download pode ser ignorada.

In [4]:
# Identificador do dataset no Kaggle (retirado da URL)
dataset = "ioexception/london-cycles"

# Diretório onde os ficheiros do dataset serão descarregados e extraídos.
# Se os ficheiros CSV foram descarregados manualmente, colocá-los/extraí-los neste mesmo diretório.
output_dir = Path("archive/")

# Verificar se o diretório archive já existe e contém ficheiros
if output_dir.exists() and any(output_dir.iterdir()):
    answer = input(
        f"O diretório '{output_dir}' já existe e contém ficheiros. "
        "Deseja apagar o seu conteúdo e descarregar o dataset novamente? (yes/no): "
    ).strip().lower()

    if answer not in ["yes", "y"]:
        print("Download ignorado. Os ficheiros existentes foram mantidos.")
    else:
        shutil.rmtree(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        # Solicitar o token da API do Kaggle sem o guardar no notebook.
        # Apenas necessário ao descarregar via API do Kaggle.
        os.environ["KAGGLE_API_TOKEN"] = getpass("Introduza o seu token da API do Kaggle: ")

        # Instalar o pacote Kaggle se necessário
        subprocess.run(["pip", "install", "-q", "kaggle"], check=True)

        # Descarregar e extrair o dataset para o diretório archive
        subprocess.run([
            "kaggle", "datasets", "download",
            "-d", dataset,
            "-p", str(output_dir),
            "--unzip"
        ], check=True)

        print(f"Dataset descarregado para: {output_dir}")

else:
    output_dir.mkdir(parents=True, exist_ok=True)
    os.environ["KAGGLE_API_TOKEN"] = getpass("Introduza o seu token da API do Kaggle: ")
    subprocess.run(["pip", "install", "-q", "kaggle"], check=True)
    subprocess.run([
        "kaggle", "datasets", "download",
        "-d", dataset,
        "-p", str(output_dir),
        "--unzip"
    ], check=True)

    print(f"Dataset descarregado para: {output_dir}")

Download ignorado. Os ficheiros existentes foram mantidos.


---

## **1. Bikes Data**

In [5]:
# Ler apenas os ficheiros CSV diários de disponibilidade de bicicletas do diretório archive.
# O pathGlobFilter evita carregar ficheiros de metadados das estações (ex: stations-YYYY-MM-DD.csv).
df_bikes = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("pathGlobFilter", "20??-??-??.csv") \
    .load("archive/")

In [6]:
# Validação básica dos dados de disponibilidade de bicicletas ingeridos
df_bikes.printSchema()
df_bikes.show(5, truncate=False)

root
 |-- query_time: timestamp (nullable = true)
 |-- place_id: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- bikes: integer (nullable = true)
 |-- empty_docks: integer (nullable = true)
 |-- docks: integer (nullable = true)

+--------------------------+------------+---------+---------+-----+-----------+-----+
|query_time                |place_id    |lat      |lon      |bikes|empty_docks|docks|
+--------------------------+------------+---------+---------+-----+-----------+-----+
|2023-09-09 00:18:51.756993|BikePoints_1|51.529163|-0.10997 |3    |16         |19   |
|2023-09-09 00:18:51.756993|BikePoints_2|51.499606|-0.197574|15   |20         |37   |
|2023-09-09 00:18:51.756993|BikePoints_3|51.521283|-0.084605|1    |31         |32   |
|2023-09-09 00:18:51.756993|BikePoints_4|51.530059|-0.120973|15   |6          |23   |
|2023-09-09 00:18:51.756993|BikePoints_5|51.49313 |-0.156876|15   |11         |27   |
+--------------------------+---

In [7]:
# Estatísticas descritivas para as colunas numéricas
df_bikes.describe().show()

+-------+-------------+------------------+--------------------+------------------+-----------------+-----------------+
|summary|     place_id|               lat|                 lon|             bikes|      empty_docks|            docks|
+-------+-------------+------------------+--------------------+------------------+-----------------+-----------------+
|  count|     35049957|          35049957|            35049957|          35049957|         35049957|         35049957|
|   mean|         NULL| 51.50493339769664|-0.12727754386892887|12.385030286913048|13.81485141336978|26.43779802639986|
| stddev|         NULL|0.2266007310575489| 0.05498382285113036|  9.29354390969849|9.785066817444491|8.840859972377896|
|    min| BikePoints_1|               0.0|           -0.236769|                 0|                0|                0|
|    max|BikePoints_99|         51.549369|                 0.0|                64|               64|               64|
+-------+-------------+------------------+------

**Nota: O formato Parquet melhora o desempenho e a eficiência de armazenamento porque é comprimido,**
**preserva o schema e permite ao Spark ler apenas as colunas necessárias para a análise.**

In [8]:
# Guardar o DataFrame como dataset Parquet para as etapas seguintes do projeto.

parquet_path = Path("data/london-bicycle")

should_write = True

if parquet_path.exists():
    answer = input(
        f"O dataset Parquet já existe em '{parquet_path}'. "
        "Deseja substituí-lo? (yes/no): "
    ).strip().lower()

    should_write = answer in ["yes", "y"]

if should_write:
    df_bikes.write.mode("overwrite").parquet(str(parquet_path))
    print(f"Dataset Parquet guardado em: {parquet_path}")
else:
    print("Gravação Parquet ignorada.")

Gravação Parquet ignorada.


---

## **2. Station Data**

In [9]:

archive_dir = Path("archive/")

station_files = sorted([
    file for file in archive_dir.glob("stations-*.csv")
    if file.stat().st_size > 10
])

print(f"Ficheiros de estações válidos: {len(station_files)}")


Ficheiros de estações válidos: 486


In [10]:
# Uniformizar os nomes das colunas
canonical_station_columns = [
    "place_id",
    "common_name",
    "installed",
    "temporary",
    "install_date",
    "terminal_name",
    "locked",
    "removal_date",
    "snapshot_date"
]

def read_station_file(path):
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "false") \
        .load(str(path))

    # Normalizar os nomes das colunas (remover espaços e converter para minúsculas)
    for old_col in df.columns:
        df = df.withColumnRenamed(old_col, old_col.strip().lower())

    # Adicionar colunas em falta como nulo (string) para garantir o mesmo schema em todos os DataFrames
    for col_name in canonical_station_columns:
        if col_name not in df.columns and col_name != "snapshot_date":
            df = df.withColumn(col_name, F.lit(None).cast(StringType()))

    # Adicionar a data do snapshot (como string) extraída do nome do ficheiro
    df = df.withColumn(
        "snapshot_date",
        F.regexp_extract(F.lit(path.name), r"\d{4}-\d{2}-\d{2}", 0)
    )

    return df.select(canonical_station_columns)

In [11]:
# Ler todos os ficheiros de estações para uma lista de DataFrames
station_dfs = [read_station_file(path) for path in station_files]

# Unir todos os DataFrames de estações num único DataFrame. O unionByName com allowMissingColumns=True
# garante que as colunas são correspondidas pelo nome e as colunas em falta são preenchidas com nulos.
df_stations = reduce(
    lambda left, right: left.unionByName(right, allowMissingColumns=True),
    station_dfs
)

df_stations.printSchema()
df_stations.show(5, truncate=False)
print(f"Linhas de estações: {df_stations.count()}")

root
 |-- place_id: string (nullable = true)
 |-- common_name: string (nullable = true)
 |-- installed: string (nullable = true)
 |-- temporary: string (nullable = true)
 |-- install_date: string (nullable = true)
 |-- terminal_name: string (nullable = true)
 |-- locked: string (nullable = true)
 |-- removal_date: string (nullable = true)
 |-- snapshot_date: string (nullable = false)

+-------------+-----------------------------------+---------+---------+-------------+-------------+------+------------+-------------+
|place_id     |common_name                        |installed|temporary|install_date |terminal_name|locked|removal_date|snapshot_date|
+-------------+-----------------------------------+---------+---------+-------------+-------------+------+------------+-------------+
|BikePoints_85|Tanner Street, Bermondsey          |true     |false    |1279026060000|000994       |false |NULL        |2022-05-20   |
|BikePoints_86|Sancroft Street, Vauxhall          |true     |false    |12790

In [12]:
# Configurações do MongoDB
mongo_uri = "mongodb://mongodb:27017"
mongo_db = "london_bikes"
stations_collection = "stations_raw"

# Escrever os dados das estações no MongoDB
df_stations.write \
    .format("mongodb") \
    .option("spark.mongodb.connection.uri", mongo_uri) \
    .option("spark.mongodb.database", mongo_db) \
    .option("spark.mongodb.collection", stations_collection) \
    .mode("overwrite") \
    .save()